# 🎬 LTX-2 Video Generator — Free Google Colab UI

Runs the LTX-2 prompt-to-video Gradio UI on Colab's **free T4 GPU**.

## ⚠️ Free Colab is _barely_ big enough

The 22 B-parameter LTX-2 model and the Gemma-3 text encoder together are **~30 GB**. Free Colab's disk is ~110 GB, but ~25 GB of that is already used by the system. We have to be careful or you'll see `Disk: 109/112 GB` and the launch will hang.

What this notebook does to fit:

- Uses the **one-stage** pipeline (skips the spatial upsampler — saves a download and a model load).
- Uses `--quantization fp8-cast` (works on the free T4's 16 GB VRAM).
- Caches weights with `hf_hub_download` (hard-links inside `~/.cache/huggingface`, no double-copy).
- Runs with `expandable_segments:True`.

## Steps

1. `Runtime → Change runtime type → T4 GPU` (free).
2. Run all cells top-to-bottom.
3. Cell 6 prints a public `https://*.gradio.live` URL — open it in your browser.

> If you've already filled your disk in a previous attempt, run **cell 0** below first to clean up.

## 0. (If retrying) Free up disk space

Wipes any previous downloads from a failed run. Skip on a fresh runtime.

In [ ]:
!rm -rf /content/models /root/.cache/huggingface /root/.cache/uv
!df -h /content

## 1. Check the GPU

In [ ]:
!nvidia-smi || echo 'No GPU detected. Go to Runtime > Change runtime type > T4 GPU.'

## 2. Clone the repo

In [ ]:
import os
if not os.path.exists('/content/LTX-2'):
    !git clone --depth 1 https://github.com/Lightricks/LTX-2.git /content/LTX-2
%cd /content/LTX-2
!pwd

## 3. Drop in the UI files

We add `app.py` + `app_pipeline.py` from this fork (the same files you have locally).

In [ ]:
# Pull the UI files from the fork that has them.
FORK = 'https://raw.githubusercontent.com/debzody/LTX-2/main'
!curl -sSL {FORK}/app.py -o app.py
!curl -sSL {FORK}/app_pipeline.py -o app_pipeline.py
!ls -la app.py app_pipeline.py

## 4. Install dependencies

Installs LTX-2 + Gradio. Takes ~5 minutes the first time.

In [ ]:
!pip install -q uv
!uv pip install --system -e packages/ltx-core -e packages/ltx-pipelines
!uv pip install --system 'gradio>=4.44' 'huggingface_hub[cli]'

## 5a. Hugging Face login (required — Gemma is gated)

The Gemma-3 text encoder is a **gated** model. You need to do this **once**:

1. Open <https://huggingface.co/google/gemma-3-12b-it-qat-q4_0-unquantized> in a new tab.
2. Click **Agree and access repository** (one-time, free, instant approval).
3. Go to <https://huggingface.co/settings/tokens> and create a token with **Read** permission. Copy it.
4. Run the cell below and paste the token when prompted (it stays only in this Colab session).

> 💡 If you already saved a `HF_TOKEN` in Colab's left-side **🔑 Secrets** panel, the cell will pick it up automatically — no prompt.

In [ ]:
import os
from huggingface_hub import login, whoami

token = None
# Try Colab secrets first.
try:
    from google.colab import userdata
    try:
        token = userdata.get('HF_TOKEN')
    except Exception:
        token = None
except ImportError:
    pass
# Fall back to env var.
if not token:
    token = os.environ.get('HF_TOKEN')
# Last resort: prompt.
if not token:
    from getpass import getpass
    print('Paste your Hugging Face read token (starts with hf_...) and press Enter:')
    token = getpass('HF token: ').strip()

if not token:
    raise RuntimeError('No HF token provided. See cell 5a description.')

login(token=token, add_to_git_credential=False)
os.environ['HF_TOKEN'] = token
print('Logged in as:', whoami()['name'])

## 5b. Download the model weights

We grab the **distilled** 22B checkpoint, the spatial upsampler, and the Gemma text encoder. ~25 GB total, downloaded straight to Colab's local disk.

If you get `GatedRepoError` here, you skipped step 5a — go back, accept the Gemma terms on the HF website, paste a valid token, and re-run.

In [ ]:
import os
from huggingface_hub import hf_hub_download, snapshot_download, login, whoami
from huggingface_hub.utils import GatedRepoError

# --- Make sure we're authenticated. Try Colab secret -> env var -> prompt. ---
def _ensure_hf_login():
    try:
        whoami()
        return  # already logged in
    except Exception:
        pass
    token = None
    try:
        from google.colab import userdata
        try:
            token = userdata.get('HF_TOKEN')
        except Exception:
            token = None
    except ImportError:
        pass
    if not token:
        token = os.environ.get('HF_TOKEN')
    if not token:
        from getpass import getpass
        print('No Hugging Face login found. Paste a read-token (https://huggingface.co/settings/tokens):')
        token = getpass('HF token: ').strip()
    if not token:
        raise RuntimeError('Aborted: no HF token provided. See cell 5a description above.')
    login(token=token, add_to_git_credential=False)
    os.environ['HF_TOKEN'] = token
    print('Logged in as:', whoami()['name'])

_ensure_hf_login()

# IMPORTANT: download into HF cache only (no local_dir copy). This keeps disk
# usage at ~30 GB instead of ~60 GB. The training code accepts the cache path.
print('Downloading distilled LTX-2 checkpoint (~22 GB)...')
ckpt = hf_hub_download(
    repo_id='Lightricks/LTX-2.3',
    filename='ltx-2.3-22b-distilled-1.1.safetensors',
)
print('  ', ckpt)

print('Downloading Gemma text encoder (~7 GB)...')
try:
    gemma = snapshot_download(
        repo_id='google/gemma-3-12b-it-qat-q4_0-unquantized',
        allow_patterns=['*.json', '*.safetensors', '*.model', 'tokenizer*', 'special_tokens*'],
    )
    print('  ', gemma)
except GatedRepoError as e:
    raise SystemExit(
        '\n\n❌ Gemma access denied. Two things must be true:\n'
        '   1) Your HF account has accepted the license at:\n'
        '      https://huggingface.co/google/gemma-3-12b-it-qat-q4_0-unquantized\n'
        '      (click "Agree and access repository")\n'
        '   2) The token you used belongs to that same account and has Read permission.\n\n'
        f'Original error: {e}'
    )

# Persist the resolved paths for the launch cell.
import json
with open('/content/_paths.json', 'w') as f:
    json.dump({'ckpt': ckpt, 'gemma': gemma}, f)

print('\nAll set. Free disk:')
!df -h /content

## 6. Launch the UI with a public link

On the free T4 (16 GB) we use `--quantization fp8-cast` + `expandable_segments:True`.

In [ ]:
import os, json
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

with open('/content/_paths.json') as f:
    paths = json.load(f)

ckpt = paths['ckpt']
gemma = paths['gemma']
print('Using checkpoint:', ckpt)
print('Using gemma root:', gemma)

# We launch with --quantization fp8-cast for the free T4 (16 GB VRAM).
# We do NOT pass --spatial-upsampler-path or --distilled-lora — the UI's
# `one-stage` pipeline doesn't need them, which saves a lot of disk + memory.
!python app.py \
    --checkpoint-path "{ckpt}" \
    --gemma-root "{gemma}" \
    --quantization fp8-cast \
    --output-dir /content/outputs \
    --host 0.0.0.0 --port 7860 \
    --share

Look for a line like:

```
Running on public URL: https://abcdef1234.gradio.live
```

Click that — it's your UI.

### Tips for free-tier T4

- In the UI, the **Pipeline** dropdown defaults to **`one-stage`** — keep it there for the free T4. (The `two-stage` and `distilled` options need the upsampler, which we skipped.)
- Keep **width / height** ≤ 512 and **frames** ≤ 65.
- First generation loads weights to GPU (1–2 min). Subsequent ones are faster.
- Colab disconnects after ~12 hours of idle. Re-run cell 6 to restart the UI; the cached weights stay on disk for the session.
- Generated MP4s are written to `/content/outputs`. Download them from the Files tab on the left.

## (Optional) 7. Stop the server

Use **Runtime → Interrupt execution** (the ⏹ button), or run:

In [ ]:
!pkill -f 'python app.py' || true